In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.sparse as sp
import os

# ── Paths ────────────────────────────────────────────────────────────
OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/"
           "niche_biology")
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'hc_ward_k10'
                                                                                                                            
CELL_TYPE_ORDER = [                                                                                                                                    
      'epithelial_luminal', 'epithelial_basal', 'epithelial_club',                                                                                       
      'epithelial_hillock', 'epithelial_neuroendocrine',                                                                                                 
      'smooth_muscle', 'fibroblast', 'endothelial',                                                                                                      
      'macrophage_group_1', 'macrophage_group_2', 'monocyte', 'dendritic', 'mast',                                                                       
      'natural_killer_cd56_bright', 'natural_killer_cd56_dim',
      'b', 'plasma',                                                                                                                                     
      't_cd4_naive_central_memory', 't_cd4_tissue_resident_memory',
      't_cd8_tissue_resident_memory', 't_cd8_cytotoxic', 't_regulatory',                                                                                 
  ]              

# ── Load files ───────────────────────────────────────────────────────
adata_mean = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

adata_vis = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/spatial_mapped_with_posterior.h5ad"
)

# ── Extract raw mean abundances ──────────────────────────────────────
mean_abundances = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
mean_abundances.columns = [col.replace('meanscell_abundance_w_sf_', '')
                            for col in mean_abundances.columns]
print("Mean abundances shape:", mean_abundances.shape)
print("Cell types:", mean_abundances.columns.tolist())

# ── Build proportion matrix ──────────────────────────────────────────
cluster_proportions = pd.DataFrame(
    mean_abundances.values,
    index=adata_mean.obs.index,
    columns=mean_abundances.columns
)

# Row-normalise so each spot sums to 1
cluster_proportions = cluster_proportions.div(
    cluster_proportions.sum(axis=1), axis=0)

# Add niche label
cluster_proportions['niche'] = adata_mean.obs[CLUSTER_KEY].values

# Mean proportion per niche across all spots
niche_means = cluster_proportions.groupby('niche')[mean_abundances.columns].mean()
niche_means.index = 'Niche_' + niche_means.index.astype(str)

print("\nNiche means shape:", niche_means.shape)
print(niche_means.round(3))

# ── Clustermap ───────────────────────────────────────────────────────               
g = sns.clustermap(
      niche_means.T.loc[CELL_TYPE_ORDER],
      row_cluster=False,                                                                                                                                 
      col_cluster=True,
      cmap='mako_r',                                                                                                                                     
      standard_scale=1,
      figsize=(12, 10),
      xticklabels=True,
      yticklabels=True,                                                                                                                                  
      linewidths=0.3,
      linecolor='white',                                                                                                                                 
      dendrogram_ratio=(0.0, 0.15),
      cbar_pos=(1.02, 0.3, 0.02, 0.4),
      cbar_kws={'label': 'Row-normalised proportion\n(0=min, 1=max per cell type)'}                                                                      
  )

g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=45, ha='right', fontsize=10
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    rotation=0, fontsize=10
)

g.fig.suptitle(
    'Niche cell type profiles — HC Ward k=10\n'
    'Mean cell type proportion per niche (all 38 slides)',
    y=1.02, fontsize=12
)

g.savefig(os.path.join(OUT_DIR, 'niche_clustermap_hc_ward_k10_simple.png'),
          dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

redo with row dendograms:

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.sparse as sp
import os

# ── Paths ────────────────────────────────────────────────────────────
OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/"
           "niche_biology")
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'hc_ward_k10'
                                                                                                                            
CELL_TYPE_ORDER = [                                                                                                                                    
      'epithelial_luminal', 'epithelial_basal', 'epithelial_club',                                                                                       
      'epithelial_hillock', 'epithelial_neuroendocrine',                                                                                                 
      'smooth_muscle', 'fibroblast', 'endothelial',                                                                                                      
      'macrophage_group_1', 'macrophage_group_2', 'monocyte', 'dendritic', 'mast',                                                                       
      'natural_killer_cd56_bright', 'natural_killer_cd56_dim',
      'b', 'plasma',                                                                                                                                     
      't_cd4_naive_central_memory', 't_cd4_tissue_resident_memory',
      't_cd8_tissue_resident_memory', 't_cd8_cytotoxic', 't_regulatory',                                                                                 
  ]              

# ── Load files ───────────────────────────────────────────────────────
adata_mean = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

adata_vis = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/spatial_mapped_with_posterior.h5ad"
)

# ── Extract raw mean abundances ──────────────────────────────────────
mean_abundances = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
mean_abundances.columns = [col.replace('meanscell_abundance_w_sf_', '')
                            for col in mean_abundances.columns]
print("Mean abundances shape:", mean_abundances.shape)
print("Cell types:", mean_abundances.columns.tolist())

# ── Build proportion matrix ──────────────────────────────────────────
cluster_proportions = pd.DataFrame(
    mean_abundances.values,
    index=adata_mean.obs.index,
    columns=mean_abundances.columns
)

# Row-normalise so each spot sums to 1
cluster_proportions = cluster_proportions.div(
    cluster_proportions.sum(axis=1), axis=0)

# Add niche label
cluster_proportions['niche'] = adata_mean.obs[CLUSTER_KEY].values

# Mean proportion per niche across all spots
niche_means = cluster_proportions.groupby('niche')[mean_abundances.columns].mean()
niche_means.index = 'Niche_' + niche_means.index.astype(str)

print("\nNiche means shape:", niche_means.shape)
print(niche_means.round(3))

# ── Clustermap ───────────────────────────────────────────────────────               
g = sns.clustermap(
      niche_means.T,
      row_cluster=True,                                                                                                                                 
      col_cluster=True,
      cmap='mako_r',                                                                                                                                     
      standard_scale=1,
      figsize=(12, 10),
      xticklabels=True,
      yticklabels=True,                                                                                                                                  
      linewidths=0.3,
      linecolor='white',                                                                                                                                 
      dendrogram_ratio=(0.15, 0.15),
      cbar_pos=(1.02, 0.3, 0.02, 0.4),
      cbar_kws={'label': 'Row-normalised proportion\n(0=min, 1=max per cell type)'}                                                                      
  )

g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=45, ha='right', fontsize=10
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    rotation=0, fontsize=10
)

g.fig.suptitle(
    'Niche cell type profiles — HC Ward k=10\n'
    'Mean cell type proportion per niche (all 38 slides)',
    y=1.02, fontsize=12
)

g.savefig(os.path.join(OUT_DIR, 'niche_clustermap_hc_ward_k10_simple.png'),
          dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")